# LSTM Question Generator

In [ ]:
# Cell 1: Imports
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import random

print('Imports OK - Ready for Visual Data')

Imports OK - Ready for Visual Data


In [ ]:
# Cell 2: Config
MAX_CONTEXT_LEN  = 20
MAX_QUESTION_LEN = 15
VOCAB_SIZE_LIMIT = 2000  # Smaller vocab because visual text is simple!
EMBEDDING_DIM    = 256
LSTM_UNITS       = 512
BATCH_SIZE       = 128   # Lowered slightly for smoother training
EPOCHS           = 15

print('Config set.')

Config set.


In [ ]:
import json
import re
from collections import Counter

with open('generated_captions.json', 'r', encoding='utf-8') as f:
    captions_data = json.load(f)

def all_questions_for_caption(caption):
    """Generate every possible valid question for a caption."""
    c = caption.lower().strip()
    words = c.split()
    word_set = set(words)
    questions = []

    people  = ['man','woman','boy','girl','child','person','player','baby','kid','skier','surfer','rider','biker']
    animals = ['dog','cat','horse','bird','cow','sheep','elephant','bear','zebra','giraffe','duck','rabbit']
    actions = ['running','jumping','sitting','standing','walking','playing','holding','wearing',
               'eating','riding','climbing','swimming','dancing','throwing','laughing','lying',
               'looking','carrying','catching','kicking','hitting','chasing','leaping','racing',
               'surfing','skiing','snowboarding','skateboarding','cycling','posing','smiling']
    colors  = ['red','blue','green','yellow','white','black','orange','purple','pink','brown','gray']
    locs    = {'beach':'the beach','park':'the park','water':'the water','field':'the field',
               'street':'the street','snow':'the snow','pool':'the pool','mountain':'the mountain',
               'forest':'the forest','grass':'the grass','road':'the road','kitchen':'the kitchen',
               'court':'the court','ocean':'the ocean','lake':'the lake','river':'the river',
               'hill':'the hill','yard':'the yard','track':'the track','sidewalk':'the sidewalk',
               'sand':'the sand','air':'the air','wall':'the wall','rock':'the rock'}
    objects = ['ball','bike','bicycle','skateboard','surfboard','kite','umbrella','hat',
               'shirt','jacket','bag','sign','flag','frisbee','racket','bat','stick',
               'rope','leash','helmet','glove','shoe','chair','bench']

    # Find what's in the caption
    found_people  = [p for p in people  if p in word_set]
    found_animals = [a for a in animals if a in word_set]
    found_actions = [a for a in actions if a in c]
    found_colors  = [col for col in colors if col in word_set]
    found_locs    = {loc: phrase for loc, phrase in locs.items() if loc in word_set}
    found_objects = [o for o in objects if o in word_set]

    subjects = found_people + found_animals

    # Generate every valid combination
    for subj in subjects:
        questions.append(f"what is the {subj} doing")
        questions.append(f"what is happening with the {subj}")

        for act in found_actions:
            questions.append(f"what is the {subj} {act}")
            questions.append(f"why is the {subj} {act}")
            for loc_phrase in found_locs.values():
                questions.append(f"where is the {subj} {act}")
                questions.append(f"what is the {subj} {act} on")

        for loc_phrase in found_locs.values():
            questions.append(f"where is the {subj}")
            questions.append(f"what is the {subj} doing at {loc_phrase}")

        for col in found_colors:
            questions.append(f"what is the {subj} wearing")
            questions.append(f"what color is the {subj} wearing")

        for obj in found_objects:
            questions.append(f"what is the {subj} doing with the {obj}")
            questions.append(f"what is the {subj} holding")

    # Location-only questions
    for loc_phrase in found_locs.values():
        questions.append(f"what is happening at {loc_phrase}")
        questions.append(f"where is this scene taking place")
        for act in found_actions:
            questions.append(f"who is {act} at {loc_phrase}")

    # Action-only questions
    for act in found_actions:
        questions.append(f"who is {act}")
        questions.append(f"what is happening here")

    # Color questions (on objects only)
    for col in found_colors:
        for obj in found_objects:
            questions.append(f"what color is the {obj}")
        if found_locs:
            questions.append(f"what color stands out in this scene")

    # Object questions
    for obj in found_objects:
        questions.append(f"what object is in the scene")
        questions.append(f"what is the person doing with the {obj}")

    # Count
    count_words = ['two','three','four','five','several','many','group','pair','couple']
    for cnt in count_words:
        if cnt in word_set:
            questions.append(f"how many people are in the image")
            questions.append(f"how many subjects are visible")
            break

    # Always add these generic ones for variety
    questions.append("what is the main subject of this image")
    questions.append("what is the overall scene")

    # Deduplicate while preserving order
    seen = set()
    unique = []
    for q in questions:
        if q not in seen:
            seen.add(q)
            unique.append(q)

    return unique

# Build pairs — every question for every caption
pairs = []
for img_path, caption in captions_data.items():
    for q in all_questions_for_caption(caption):
        pairs.append((caption, q))

# Balance — cap each question at 200 to avoid dominance
from collections import defaultdict
question_count = defaultdict(int)
MAX_PER_Q = 200
balanced = []
for cap, q in pairs:
    if question_count[q] < MAX_PER_Q:
        balanced.append((cap, q))
        question_count[q] += 1
pairs = balanced

# Report
q_counts = Counter(q for _, q in pairs)
print(f"Total pairs     : {len(pairs)}")
print(f"Unique questions: {len(q_counts)}")
print(f"\nTop 10 most common:")
for q, cnt in q_counts.most_common(10):
    pct = cnt / len(pairs) * 100
    print(f"  {pct:5.1f}%  ({cnt:4d}x)  {q}")

print(f"\nSample pairs:")
import random
for cap, q in random.sample(pairs, min(8, len(pairs))):
    print(f"  Caption : {cap}")
    print(f"  Question: {q}")
    print()

Total pairs     : 28051
Unique questions: 327

Top 10 most common:
    0.7%  ( 200x)  what is the woman doing
    0.7%  ( 200x)  what is happening with the woman
    0.7%  ( 200x)  what is the woman standing
    0.7%  ( 200x)  why is the woman standing
    0.7%  ( 200x)  what is the woman wearing
    0.7%  ( 200x)  what color is the woman wearing
    0.7%  ( 200x)  what is the woman doing with the shirt
    0.7%  ( 200x)  what is the woman holding
    0.7%  ( 200x)  who is standing
    0.7%  ( 200x)  what is happening here

Sample pairs:
  Caption : a man in a red shirt is riding a bike on a dirt path
  Question: what is the person doing with the bike

  Caption : a man in a red shirt is riding a horse on a track
  Question: what is the man doing at the track

  Caption : a man in a wetsuit is surfing on a wave
  Question: who is surfing

  Caption : a woman in a red shirt is standing in front of a building
  Question: what is the woman doing with the shirt

  Caption : a little girl i

In [ ]:
# Cell 3b: Remove nonsense questions

bad_patterns = [
    "what is the woman standing",      # incomplete not a real question
    "what is the man standing",
    "what is the boy standing",
    "what is the girl standing",
    "what is the dog standing",
    "what is the horse doing with the shirt",  # shirt belongs to rider not horse
    "what is the dog doing with the shirt",
    "what is the bird doing with the shirt",
    "what is the boy doing with the shirt",    # shirt isn't what boy is doing
    "what is the girl doing with the shirt",
    "what is the man doing with the shirt",
    "what is the woman doing with the shirt",
]

pairs = [(cap, q) for cap, q in pairs if q not in bad_patterns]

q_counts = Counter(q for _, q in pairs)
print(f"Total pairs     : {len(pairs)}")
print(f"Unique questions: {len(q_counts)}")
print(f"Top question    : {q_counts.most_common(1)[0][1]/len(pairs)*100:.1f}%")
print("Ready to train!")

Total pairs     : 26569
Unique questions: 316
Top question    : 0.8%
Ready to train!


In [ ]:
# Cell 4: Clean Text
def clean(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s\?]", '', text) # Simple cleaning
    text = re.sub(r'\s+', ' ', text).strip()
    return text

contexts  = [clean(c) for c, _ in pairs]
questions = ['startseq ' + clean(q) + ' endseq' for _, q in pairs]

In [ ]:
# Cell 5: Tokenizer
tokenizer = Tokenizer(num_words=VOCAB_SIZE_LIMIT, oov_token='<unk>')
tokenizer.fit_on_texts(contexts + questions)

word_to_idx = tokenizer.word_index
idx_to_word = tokenizer.index_word
vocab_size  = min(len(word_to_idx) + 1, VOCAB_SIZE_LIMIT + 1)
print(f'Actual Vocabulary Size: {vocab_size}')

Actual Vocabulary Size: 157


In [ ]:
# Cell 6: Train/Val Split
val_split_idx = int(0.9 * len(pairs))
train_pairs = pairs[:val_split_idx]
val_pairs   = pairs[val_split_idx:]

def calculate_steps(dataset):
    total = 0
    for _, q in dataset:
        q_len = len(clean(q).split()) + 2
        total += min(q_len, MAX_QUESTION_LEN) - 1
    return total

train_total = calculate_steps(train_pairs)
val_total   = calculate_steps(val_pairs)

steps_per_epoch = max(1, train_total // BATCH_SIZE)
val_steps       = max(1, val_total // BATCH_SIZE)
print(f"Steps per epoch: {steps_per_epoch}")

Steps per epoch: 1240


In [ ]:
# Cell 7: Encoder-Decoder Model
# Encoder
enc_input = Input(shape=(MAX_CONTEXT_LEN,), name='enc_input')
enc_embed = Embedding(vocab_size, EMBEDDING_DIM, mask_zero=True)(enc_input)
_, enc_h, enc_c = LSTM(LSTM_UNITS, return_state=True)(enc_embed)

# Decoder
dec_input = Input(shape=(MAX_QUESTION_LEN,), name='dec_input')
dec_embed = Embedding(vocab_size, EMBEDDING_DIM, mask_zero=True)(dec_input)
dec_out   = LSTM(LSTM_UNITS, return_sequences=False)(dec_embed, initial_state=[enc_h, enc_c])

# Output
dec_out = Dropout(0.3)(dec_out)
dec_out = Dense(vocab_size, activation='softmax')(dec_out)

model = Model(inputs=[enc_input, dec_input], outputs=dec_out)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ enc_input           │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_input           │ (None, 15)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_16        │ (None, 20, 256)   │     40,192 │ enc_input[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_16        │ (None, 20)        │          0 │ enc_input[0][0]   │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_17        │ (None, 15, 256)   │     40,192 │ dec_input[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_16 (LSTM)      │ [(None, 512),     │  1,574,912 │ embedding_16[0][… │
│                     │ (None, 512),      │            │ not_equal_16[0][… │
│                     │ (None, 512)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_17 (LSTM)      │ (None, 512)       │  1,574,912 │ embedding_17[0][… │
│                     │                   │            │ lstm_16[0][1],    │
│                     │                   │            │ lstm_16[0][2]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 512)       │          0 │ lstm_17[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 157)       │     80,541 │ dropout_8[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,310,749 (12.63 MB)

 Trainable params: 3,310,749 (12.63 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Cell 8: Data Generator
def pair_generator(dataset_pairs, tokenizer_obj, batch_size):
    enc_batch, dec_batch, out_batch = [], [], []
    while True:
        idx = np.random.permutation(len(dataset_pairs))
        for i in idx:
            c_text, q_text = dataset_pairs[i]
            ctx_seq = tokenizer_obj.texts_to_sequences([clean(c_text)])[0][:MAX_CONTEXT_LEN]
            q_seq   = tokenizer_obj.texts_to_sequences(['startseq ' + clean(q_text) + ' endseq'])[0][:MAX_QUESTION_LEN]

            if len(q_seq) < 2: continue
            ctx_padded = pad_sequences([ctx_seq], maxlen=MAX_CONTEXT_LEN, padding='post')[0]

            for j in range(1, len(q_seq)):
                dec_in = pad_sequences([q_seq[:j]], maxlen=MAX_QUESTION_LEN, padding='post')[0]
                enc_batch.append(ctx_padded)
                dec_batch.append(dec_in)
                out_batch.append(q_seq[j])

                if len(enc_batch) == batch_size:
                    yield ((np.array(enc_batch), np.array(dec_batch)), np.array(out_batch))
                    enc_batch, dec_batch, out_batch = [], [], []

output_signature = (
    (tf.TensorSpec(shape=(None, MAX_CONTEXT_LEN), dtype=tf.int32),
     tf.TensorSpec(shape=(None, MAX_QUESTION_LEN), dtype=tf.int32)),
    tf.TensorSpec(shape=(None,), dtype=tf.int32)
)

train_ds = tf.data.Dataset.from_generator(lambda: pair_generator(train_pairs, tokenizer, BATCH_SIZE), output_signature=output_signature)
val_ds   = tf.data.Dataset.from_generator(lambda: pair_generator(val_pairs, tokenizer, BATCH_SIZE), output_signature=output_signature)

In [ ]:
# Cell 9: Train the Model
callbacks = [
    EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1)
]

print("Starting training on clean visual data...")
history = model.fit(train_ds, steps_per_epoch=steps_per_epoch, validation_data=val_ds, validation_steps=val_steps, epochs=EPOCHS, callbacks=callbacks)

Starting training on clean visual data...
Epoch 1/15
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 31s 23ms/step - accuracy: 0.6603 - loss: 1.1446 - val_accuracy: 0.7319 - val_loss: 0.7394 - learning_rate: 0.0010
Epoch 2/15
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 27s 22ms/step - accuracy: 0.8097 - loss: 0.5084 - val_accuracy: 0.8288 - val_loss: 0.4445 - learning_rate: 0.0010
Epoch 3/15
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 28s 22ms/step - accuracy: 0.8284 - loss: 0.4334 - val_accuracy: 0.8369 - val_loss: 0.3948 - learning_rate: 0.0010
Epoch 4/15
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 28s 23ms/step - accuracy: 0.8311 - loss: 0.4186 - val_accuracy: 0.8467 - val_loss: 0.3735 - learning_rate: 0.0010
Epoch 5/15
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 28s 22ms/step - accuracy: 0.8332 - loss: 0.4064 - val_accuracy: 0.8583 - val_loss: 0.3699 - learning_rate: 0.0010
Epoch 6/15
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 28s 23ms/step - accuracy: 0.8340 - loss: 0.4014 - val_accuracy: 0.8547 - val_loss: 0.3680 - learning_rate: 0.0010
Epoch 7/15
1240/1240 ━

In [120]:
def generate_visual_question(caption: str, beam_width: int = 5) -> str:
    clean_cap = clean(caption)

    ctx_seq = tokenizer.texts_to_sequences([clean_cap])[0]
    ctx_seq = pad_sequences(
        [ctx_seq[:MAX_CONTEXT_LEN]],
        maxlen=MAX_CONTEXT_LEN,
        padding='post'
    )

    end_idx   = word_to_idx.get('endseq', 0)
    start_idx = word_to_idx.get('startseq', 1)

    beams     = [(0.0, [start_idx])]
    completed = []

    # Extract key words from caption to guide generation
    cap_words = set(clean_cap.lower().split())

    # Question starters to encourage
    starters = ['what','who','where']

    for step in range(MAX_QUESTION_LEN):
        if not beams:
            break
        candidates = []

        beam_seqs = np.array([
            pad_sequences([s], maxlen=MAX_QUESTION_LEN, padding='post')[0]
            for _, s in beams
        ])
        beam_ctx  = np.repeat(ctx_seq, len(beams), axis=0)
        preds     = model.predict([beam_ctx, beam_seqs], verbose=0)

        for i, (score, seq) in enumerate(beams):
            top_indices = np.argsort(preds[i])[-beam_width * 3:]

            for idx in top_indices:
                word    = idx_to_word.get(int(idx), '')
                if not word:
                    continue

                log_prob = np.log(preds[i, idx] + 1e-10)

                # Force question word at step 0
                if step == 0:
                    if word in starters:
                        log_prob += 5.0
                    else:
                        log_prob -= 10.0

                # Penalize repeating the same word twice
                prev_words = [idx_to_word.get(w, '') for w in seq]
                if word in prev_words and word not in {'is','the','a','in','on','at','of'}:
                    log_prob -= 8.0

                # Reward caption-relevant words
                if word in cap_words and word not in {'a','the','in','on','is','at','of','with','and'}:
                    log_prob += 3.0

                # Don't end too early
                if int(idx) == end_idx and len(seq) < 5:
                    log_prob -= 15.0

                new_score = score - log_prob
                new_seq   = seq + [int(idx)]

                if int(idx) == end_idx:
                    completed.append((new_score, new_seq))
                else:
                    candidates.append((new_score, new_seq))

        beams = sorted(candidates, key=lambda x: x[0])[:beam_width]

    if not completed:
        completed = beams

    best_seq     = min(completed, key=lambda x: x[0])[1]
    output_words = [
        idx_to_word.get(i, '')
        for i in best_seq[1:]
        if i not in {end_idx, start_idx, 0}
    ]

    result = ' '.join(output_words).strip()
    if not result:
        return "What is in the image?"

    # Fix grammar ensure "the" before nouns after who/what/where
    result = re.sub(r'\b(who is|what is|where is)\s+(man|woman|boy|girl|dog|cat|horse|person|child)\b',
                    lambda m: m.group(1) + ' the ' + m.group(2), result)

    result = result[0].upper() + result[1:]
    if not result.endswith('?'):
        result += '?'
    return result

In [121]:
# Cell 11: Final Inference
import json

# Load real project data
try:
    with open('generated_captions.json', 'r', encoding='utf-8') as f:
        project_captions = json.load(f)

    print(f"Successfully loaded {len(project_captions)} captions from JSON.\n")
    print('--- Results on Project Data ---')

    # Run the generator on your first 10 images
    for i, (img_path, cap) in enumerate(list(project_captions.items())[:10]):
        question = generate_visual_question(cap)
        print(f"Image   : {img_path}")
        print(f"Caption : {cap}")
        print(f"Question: {question}")
        print("-" * 45)

except FileNotFoundError:
    print("Error: 'generated_captions.json' not found. Please ensure the file is in your directory.")

Successfully loaded 8091 captions from JSON.

--- Results on Project Data ---
Image   : 1000268201_693b08cb0e.jpg
Caption : a woman in a red shirt is standing in front of a building
Question: What is the woman wearing?
---------------------------------------------
Image   : 1001773457_577c3a7d70.jpg
Caption : a dog is running through a field
Question: Who is running at the field?
---------------------------------------------
Image   : 1002674143_1b742ab4b8.jpg
Caption : a young girl in a red shirt is standing on a bench with a man in a red shirt
Question: What is the man doing with the bench?
---------------------------------------------
Image   : 1003163366_44323f5815.jpg
Caption : a man in a red shirt is standing in front of a man in a red shirt
Question: What is the man wearing?
---------------------------------------------
Image   : 1007129816_e794419615.jpg
Caption : a man in a red shirt is standing in front of a building
Question: What is the man wearing?
------------------------

In [122]:
test_captions = [
    "a black cat is sitting on a wooden chair",
    "two children are playing soccer in a park",
    "a woman in a blue jacket is skiing down a mountain",
    "a group of people are eating at a restaurant",
    "a brown dog is swimming in a lake"
]

print("=== Final Quality Check ===\n")
for cap in test_captions:
    q = generate_visual_question(cap)
    print(f"Caption : {cap}")
    print(f"Question: {q}")
    print()

=== Final Quality Check ===

Caption : a black cat is sitting on a wooden chair
Question: What is the girl sitting?

Caption : two children are playing soccer in a park
Question: Who is playing at the park?

Caption : a woman in a blue jacket is skiing down a mountain
Question: What is the woman doing at the mountain?

Caption : a group of people are eating at a restaurant
Question: What is happening at the sidewalk?

Caption : a brown dog is swimming in a lake
Question: Who is climbing at the lake?



### note: weird questions are due to some words not appearing due training (restaurnt for example)